In [ ]:
import napari
import numpy as np
from magicgui import magicgui
from skimage import filters, morphology, measure, io, restoration, exposure
from skimage.morphology import white_tophat, ball, disk
import pandas as pd
import os
from scipy import ndimage
from napari.types import ImageData, LabelsData
import matplotlib.pyplot as plt
from datetime import datetime

In [ ]:
# --------------------------
#   Set working directory
# --------------------------
project_folder = ''
os.chdir(project_folder)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# --------------------------
#   Start napari
# --------------------------
viewer = napari.Viewer()

In [ ]:
# --------------------------
#   Check image shape
# --------------------------
img = viewer.layers[0].data
print(img.shape)

In [ ]:
# -------------------------------
#   Check pixel size in Fiji
#   Image > Properties...
#   Set pixel size
# -------------------------------
pixel_size_xy_um = 0.1803752

In [ ]:
# ---------------------------------------------------------
#  Adjust the images to analyses if necessary (z-slices)
# ---------------------------------------------------------

ch0 = img[:, 0, :, :]
ch1 = img[:, 1, :, :]
ch2 = img[:, 2, :, :]

#viewer.add_image(ch0, name='nuclei', colormap='blue', blending='additive')
viewer.add_image(ch1, name='ch1_view', colormap='green', blending='additive')
viewer.add_image(ch2, name='ch2_view', colormap='magenta', blending='additive')

In [ ]:
# ------------------------------------------
#   Check for the layers avaliable to you 
# ------------------------------------------
existing_names = [layer.name for layer in viewer.layers]
existing_names

In [ ]:
# ---------------------------------
# Helper functions
# ---------------------------------
def append_results_to_csv(results_df, csv_path):
    write_header = not os.path.exists(csv_path)
    results_df.to_csv(csv_path, mode="a", header=write_header, index=False)


def get_valid_pixels(a, b, roi_mask=None):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    valid = np.isfinite(a) & np.isfinite(b)
    if roi_mask is not None:
        valid &= roi_mask.astype(bool)

    return a[valid], b[valid]


def pearson_corr_2d(a, b, roi_mask=None):
    x, y = get_valid_pixels(a, b, roi_mask=roi_mask)

    if len(x) < 2:
        return np.nan
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def compute_auto_thresholds_2d(a, b, roi_mask=None, method="otsu", multiplier=1.0):
    x, y = get_valid_pixels(a, b, roi_mask=roi_mask)

    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan

    if method == "otsu":
        t1 = filters.threshold_otsu(x)
        t2 = filters.threshold_otsu(y)
    elif method == "li":
        t1 = filters.threshold_li(x)
        t2 = filters.threshold_li(y)
    elif method == "yen":
        t1 = filters.threshold_yen(x)
        t2 = filters.threshold_yen(y)
    else:
        raise ValueError("threshold_method must be 'otsu', 'li', or 'yen'")

    return float(t1) * float(multiplier), float(t2) * float(multiplier)


def manders_2d(a, b, roi_mask=None, thr1=0.0, thr2=0.0):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    valid = np.isfinite(a) & np.isfinite(b)
    if roi_mask is not None:
        valid &= roi_mask.astype(bool)

    x = a[valid]
    y = b[valid]

    if len(x) == 0:
        return np.nan, np.nan

    sum_x = np.sum(x[x > thr1])
    sum_y = np.sum(y[y > thr2])

    coloc_x = np.sum(x[(x > thr1) & (y > thr2)])
    coloc_y = np.sum(y[(y > thr2) & (x > thr1)])

    m1 = coloc_x / sum_x if sum_x > 0 else np.nan
    m2 = coloc_y / sum_y if sum_y > 0 else np.nan

    return float(m1), float(m2)


def shapes_to_2d_mask(shapes_layer, yx_shape):
    """
    Combine all shapes into a single 2D ROI mask.
    This ROI is then reused for every z-slice.
    """
    if shapes_layer is None or len(shapes_layer.data) == 0:
        return None

    masks = shapes_layer.to_masks(mask_shape=yx_shape)
    if len(masks) == 0:
        return None

    roi2d = np.zeros(yx_shape, dtype=bool)
    for m in masks:
        roi2d |= m.astype(bool)

    return roi2d


def remove_layer_if_exists(name):
    if name in viewer.layers:
        viewer.layers.remove(name)

def compute_auto_thresholds_stack(a, b, roi2d=None, method="otsu", multiplier=1.0):
    """
    Compute one threshold per channel from the whole 3D stack.
    If roi2d is given, the same ROI is applied to every z slice.
    """
    import numpy as np
    from skimage import filters

    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    if a.ndim != 3 or b.ndim != 3:
        raise ValueError("Expected stacks with shape (z, y, x).")

    valid = np.isfinite(a) & np.isfinite(b)

    if roi2d is not None:
        roi3d = np.broadcast_to(roi2d, a.shape)
        valid &= roi3d

    x = a[valid]
    y = b[valid]

    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan

    if method == "otsu":
        t1 = filters.threshold_otsu(x)
        t2 = filters.threshold_otsu(y)
    elif method == "li":
        t1 = filters.threshold_li(x)
        t2 = filters.threshold_li(y)
    elif method == "yen":
        t1 = filters.threshold_yen(x)
        t2 = filters.threshold_yen(y)
    else:
        raise ValueError("threshold_method must be 'otsu', 'li', or 'yen'")

    return float(t1) * float(multiplier), float(t2) * float(multiplier)

# Here draw ROI if needed

Use shape option in napari in whichever layer you would like - it will apply to all of them

In [ ]:
# -----------------------------------------------------------------------------
# Widget
# -----------------------------------------------------------------------------
@magicgui(
    image1={"label": "Channel 1"},
    image2={"label": "Channel 2"},
    use_roi={"label": "Use ROI"},
    roi_shapes={"label": "ROI Shapes Layer"},
    threshold_mode={"choices": ["auto", "manual"], "label": "Threshold mode"},
    threshold_method={"choices": ["otsu", "li", "yen"], "label": "Auto method"},
    threshold_multiplier={"label": "Auto threshold multiplier"},
    manual_threshold1={"label": "Manual threshold ch1"},
    manual_threshold2={"label": "Manual threshold ch2"},
    show_threshold_masks={"label": "Show threshold masks"},
    export_csv={"label": "Export CSV"},
    sample_name={"label": "Sample / file name"},
    csv_path={"label": "CSV path"},
    call_button="Run 2D slice colocalization",
)
def run_2d_slice_coloc(
    image1: napari.layers.Image,
    image2: napari.layers.Image,
    use_roi: bool = False,
    roi_shapes: napari.layers.Shapes = None,
    threshold_mode: str = "auto",
    threshold_method: str = "otsu",
    threshold_multiplier: float = 1.0,
    manual_threshold1: float = 0.0,
    manual_threshold2: float = 0.0,
    show_threshold_masks: bool = True,
    export_csv: bool = False,
    sample_name: str = "",
    csv_path: str = "colocalization_2d_per_slice.csv",
):
    a = np.asarray(image1.data)
    b = np.asarray(image2.data)

    if a.shape != b.shape:
        raise ValueError("Selected image layers must have the same shape.")

    if a.ndim != 3:
        raise ValueError("This analysis expects 3D channel stacks with shape (z, y, x).")

    zdim, ydim, xdim = a.shape

    if sample_name.strip() == "":
        sample_name = image1.name

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # ROI drawn once, applied to all slices
    roi2d = None
    if use_roi:
        roi2d = shapes_to_2d_mask(roi_shapes, (ydim, xdim))
        if roi2d is None:
            raise ValueError("Use ROI is checked, but no valid ROI shape was found.")

    # One threshold pair for the whole stack
    if threshold_mode == "auto":
        thr1, thr2 = compute_auto_thresholds_stack(
            a,
            b,
            roi2d=roi2d,
            method=threshold_method,
            multiplier=threshold_multiplier,
        )
    elif threshold_mode == "manual":
        thr1 = float(manual_threshold1)
        thr2 = float(manual_threshold2)
    else:
        raise ValueError("threshold_mode must be 'auto' or 'manual'")

    print(f"Threshold image1: {thr1}")
    print(f"Threshold image2: {thr2}")

    rows = []

    # Full 3D threshold masks so you can scroll through all z slices
    stack_fg1 = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    stack_fg2 = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    stack_coloc = np.zeros((zdim, ydim, xdim), dtype=np.float32)

    for z in range(zdim):
        a2 = a[z]
        b2 = b[z]

        pcc = pearson_corr_2d(a2, b2, roi_mask=roi2d)
        m1, m2 = manders_2d(a2, b2, roi_mask=roi2d, thr1=thr1, thr2=thr2)

        n_pixels = int(roi2d.sum()) if roi2d is not None else int(np.isfinite(a2).sum())

        rows.append({
            "timestamp": timestamp,
            "sample_name": sample_name,
            "image1": image1.name,
            "image2": image2.name,
            "z_index": z,
            "used_roi": bool(use_roi),
            "threshold_mode": threshold_mode,
            "threshold_method": threshold_method if threshold_mode == "auto" else "manual",
            "threshold_multiplier": threshold_multiplier if threshold_mode == "auto" else np.nan,
            "threshold_image1": thr1,
            "threshold_image2": thr2,
            "pearson_r": pcc,
            "manders_M1": m1,
            "manders_M2": m2,
            "n_analyzed_pixels": n_pixels,
        })

        fg1 = a2 > thr1
        fg2 = b2 > thr2

        if roi2d is not None:
            fg1 &= roi2d
            fg2 &= roi2d

        stack_fg1[z] = fg1.astype(np.float32)
        stack_fg2[z] = fg2.astype(np.float32)
        stack_coloc[z] = (fg1 & fg2).astype(np.float32)

    results = pd.DataFrame(rows)
    print(results)

    print("ch1 threshold mask pixels:", int(stack_fg1.sum()))
    print("ch2 threshold mask pixels:", int(stack_fg2.sum()))
    print("coloc threshold mask pixels:", int(stack_coloc.sum()))

    # Remove previous display layers before adding new ones
    for layer_name in [
        "ROI_mask_2d",
        "ch1_threshold_mask_2d",
        "ch2_threshold_mask_2d",
        "coloc_threshold_mask_2d",
    ]:
        if layer_name in viewer.layers:
            viewer.layers.remove(layer_name)
        
    # ROI display
    if use_roi and roi2d is not None:
        roi3d = np.zeros((zdim, ydim, xdim), dtype=np.float32)
        roi3d[:] = roi2d.astype(np.float32)
        viewer.add_image(
            roi3d,
            name="ROI_mask_2d",
            colormap="white",
            opacity=0.2,
            blending="additive",
        )

    # Threshold masks for each channel + overlap
    if show_threshold_masks:
        viewer.add_image(
            stack_fg1,
            name="ch1_threshold_mask_2d",
            colormap="red",
            opacity=0.7,
            blending="additive",
        )
        viewer.add_image(
            stack_fg2,
            name="ch2_threshold_mask_2d",
            colormap="green",
            opacity=0.7,
            blending="additive",
        )
        viewer.add_image(
            stack_coloc,
            name="coloc_threshold_mask_2d",
            colormap="yellow",
            opacity=0.7,
            blending="additive",
        )

    if export_csv:
        out_path = os.path.abspath(csv_path)
        append_results_to_csv(results, out_path)
        print(f"Appended per-slice results to: {out_path}")

    return results


viewer.window.add_dock_widget(run_2d_slice_coloc, area="right")